In [1]:
import xarray as xr
import pandas as pd
import os

In [2]:
print(os.getcwd())
#Load the dataset
times_ds = xr.open_dataset('sg175_May_2025_UW_OTTER_(SG175)_timeseries.nc')
display(times_ds)

C:\Users\lydia\Seagliders


<xarray.Dataset> Size: 14MB
Dimensions:                    (gps_info: 1104, sg_data_point: 95276,
                                trajectory: 368, dive: 368)
Coordinates:
    ctd_time                   (sg_data_point) datetime64[ns] 762kB ...
    ctd_depth                  (sg_data_point) float32 381kB ...
    latitude                   (sg_data_point) float32 381kB ...
    longitude                  (sg_data_point) float32 381kB ...
  * trajectory                 (trajectory) int32 1kB 1 2 3 4 ... 366 367 368
Dimensions without coordinates: gps_info, sg_data_point, dive
Data variables: (12/57)
    gps_info_dive_number       (gps_info) int32 4kB ...
    sg_data_point_dive_number  (sg_data_point) int32 381kB ...
    log_gps_time               (gps_info) datetime64[ns] 9kB ...
    time                       (sg_data_point) datetime64[ns] 762kB ...
    pressure                   (sg_data_point) float32 381kB ...
    depth                      (sg_data_point) float32 381kB ...
    ...                         ...
    end_longitude              (dive) float32 1kB ...
    depth_avg_curr_east        (dive) float32 1kB ...
    depth_avg_curr_north       (dive) float32 1kB ...
    depth_avg_curr_qc          (dive) |S1 368B ...
    latlong_qc                 (dive) |S1 368B ...
    glider                     |S12 12B ...
Attributes: (12/47)
    project:                         May 2025 UW OTTER (SG175)
    title:                           Physical and chemical data collected fro...
    summary:                         SG175 May 2025 UW OTTER (SG175)
    source:                          Seaglider SG175
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2025-06-13T16:06:14Z
    uuid:                            2cf8eaf2-4877-11f0-80b3-07c04838153a
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6

In [3]:
#Align time with sg_data_point and apply offset
adjusted_time = pd.to_datetime(times_ds['time'].values) + pd.DateOffset(years=0, months=0, days=0)

times_ds = times_ds.assign_coords(time=('sg_data_point', adjusted_time))

#Select the relevant variables
new_times_ds = times_ds[['time', 'depth', 'latitude', 'longitude','temperature', 'salinity']] 
print(new_times_ds['time'][0])

#Convert to DataFrame and save
new_times_ds.to_dataframe().reset_index().to_csv('sg175_May_2025_UW_OTTER_(SG175)_timeseries_cleaned.csv', index=False)
new_times_ds.to_netcdf('sg175_May_2025_UW_OTTER_(SG175)_timeseries_cleaned.nc')

<xarray.DataArray 'time' ()> Size: 8B
array('2025-05-30T19:25:53.948999936', dtype='datetime64[ns]')
Coordinates:
    time       datetime64[ns] 8B 2025-05-30T19:25:53.948999936
    latitude   float32 4B ...
    longitude  float32 4B ...
    ctd_time   datetime64[ns] 8B ...
    ctd_depth  float32 4B ...


In [4]:
times_ds

<xarray.Dataset> Size: 14MB
Dimensions:                    (gps_info: 1104, sg_data_point: 95276,
                                trajectory: 368, dive: 368)
Coordinates:
    time                       (sg_data_point) datetime64[ns] 762kB 2025-05-3...
    ctd_time                   (sg_data_point) datetime64[ns] 762kB 2025-05-3...
    ctd_depth                  (sg_data_point) float32 381kB 0.505 ... 0.5818
    latitude                   (sg_data_point) float32 381kB 47.84 ... 47.83
    longitude                  (sg_data_point) float32 381kB -122.4 ... -122.4
  * trajectory                 (trajectory) int32 1kB 1 2 3 4 ... 366 367 368
Dimensions without coordinates: gps_info, sg_data_point, dive
Data variables: (12/56)
    gps_info_dive_number       (gps_info) int32 4kB ...
    sg_data_point_dive_number  (sg_data_point) int32 381kB ...
    log_gps_time               (gps_info) datetime64[ns] 9kB ...
    pressure                   (sg_data_point) float32 381kB ...
    depth                      (sg_data_point) float32 381kB 1.098 ... 0.5889
    speed_gsm                  (sg_data_point) float32 381kB ...
    ...                         ...
    end_longitude              (dive) float32 1kB ...
    depth_avg_curr_east        (dive) float32 1kB ...
    depth_avg_curr_north       (dive) float32 1kB ...
    depth_avg_curr_qc          (dive) |S1 368B ...
    latlong_qc                 (dive) |S1 368B ...
    glider                     |S12 12B ...
Attributes: (12/47)
    project:                         May 2025 UW OTTER (SG175)
    title:                           Physical and chemical data collected fro...
    summary:                         SG175 May 2025 UW OTTER (SG175)
    source:                          Seaglider SG175
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2025-06-13T16:06:14Z
    uuid:                            2cf8eaf2-4877-11f0-80b3-07c04838153a
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6

In [5]:
times_ds = xr.open_dataset('sg175_May_2025_UW_OTTER_(SG175)_timeseries.nc')

#Apply time apply offset (if needed)
adjusted_time = pd.to_datetime(times_ds['time'].values) + pd.DateOffset(years=0, months=0, days=0)

times_ds['U_DAC'] = times_ds['depth_avg_curr_east']
times_ds['V_DAC'] = times_ds['depth_avg_curr_north']

# add metadata
times_ds['U_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_east'
times_ds['V_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_north'

#Select the relevant variables
new_times_ds = times_ds[['U_DAC', 'V_DAC', 'start_time', 'end_time', 'start_latitude', 'end_latitude', 'start_longitude', 'end_longitude']]
display(new_times_ds)

#Convert to DataFrame and save
new_times_ds.to_dataframe().reset_index().to_csv('sg175_May_2025_UW_OTTER_(SG175)_DAC_timeseries_cleaned.csv', index=False)
new_times_ds.to_netcdf('sg175_May_2025_UW_OTTER_(SG175)_DAC_timeseries_cleaned.nc')


<xarray.Dataset> Size: 15kB
Dimensions:          (dive: 368)
Dimensions without coordinates: dive
Data variables:
    U_DAC            (dive) float32 1kB ...
    V_DAC            (dive) float32 1kB ...
    start_time       (dive) datetime64[ns] 3kB ...
    end_time         (dive) datetime64[ns] 3kB ...
    start_latitude   (dive) float32 1kB ...
    end_latitude     (dive) float32 1kB ...
    start_longitude  (dive) float32 1kB ...
    end_longitude    (dive) float32 1kB ...
Attributes: (12/47)
    project:                         May 2025 UW OTTER (SG175)
    title:                           Physical and chemical data collected fro...
    summary:                         SG175 May 2025 UW OTTER (SG175)
    source:                          Seaglider SG175
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2025-06-13T16:06:14Z
    uuid:                            2cf8eaf2-4877-11f0-80b3-07c04838153a
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6